# CrowdChain — سجل قراءات الحشود في المشاعر المقدسة

نموذج أولي لسجل مشترك مقاوم للتلاعب، تسجّل فيه عدة جهات (الأمن، الصحة، التشغيل) قراءات كثافة الحشود، باستخدام مبادئ البلوكتشين.

**مهمة "حلّ مشكلة بتقنية ناشئة" — نادي طويق، جامعة أم القرى، مسار التقنيات الناشئة**
إعداد: ميران عواض الدعدي

## المشكلة
في موسم الحج ترصد عدة جهات كثافة الحشود، وكل جهة تحفظ قراءاتها في نظام منفصل يمكن تعديله لاحقًا دون أثر. عند وقوع حادثة يصعب إثبات ما كانت تقوله البيانات فعلًا ومتى.

## هل نحتاج بلوكتشين فعلًا؟
- أكثر من جهة تكتب في السجل ← نعم
- تحتاج الجهات سجلًا مشتركًا واحدًا ← نعم
- توجد جهة واحدة يثق بها الجميع لتملك السجل؟ ← لا، فكل جهة طرف في أي حادثة
- إثبات أن البيانات القديمة لم تتغير مهم؟ ← نعم، هذا جوهر المساءلة

**النتيجة:** لو كانت جهة واحدة موثوقة لكفت قاعدة بيانات عادية، لكن تعدد الجهات المتساوية وغياب طرف محايد هو بالضبط ما يبرر البلوكتشين.

In [1]:
import copy
import hashlib
import json
from datetime import datetime, timezone

## 1) كتلة تحمل بيانات وهاش
كل كتلة تخزّن: رقمها، والوقت، والجهة المسجِّلة، والقراءة، وهاش الكتلة السابقة. ويُحسب هاشها الخاص (SHA-256) من جميع هذه الحقول، فأي تغيير في أي حقل يغيّر الهاش.

In [2]:
class Block:
    def __init__(self, index, agency, reading, previous_hash, timestamp=None):
        self.index = index
        self.timestamp = timestamp or datetime.now(timezone.utc).isoformat(timespec="seconds")
        self.agency = agency            # الجهة التي سجّلت القراءة
        self.reading = reading          # بيانات الحشود نفسها
        self.previous_hash = previous_hash
        self.hash = self.compute_hash()

    def compute_hash(self):
        """هاش SHA-256 لكل الحقول ما عدا الهاش نفسه."""
        content = json.dumps(
            {
                "index": self.index,
                "timestamp": self.timestamp,
                "agency": self.agency,
                "reading": self.reading,
                "previous_hash": self.previous_hash,
            },
            sort_keys=True,
            ensure_ascii=False,
        )
        return hashlib.sha256(content.encode("utf-8")).hexdigest()

    def __repr__(self):
        r = self.reading
        if "site" in r:
            body = (f"الجهة: {self.agency} | الموقع: {r['site']} | "
                    f"الكثافة: {r['density_per_m2']} شخص/م² | الحالة: {r['status']}")
        else:
            body = r.get("note", "")
        return (f"الكتلة {self.index} | {body}\n"
                f"      الهاش: {self.hash[:12]}...   السابق: {self.previous_hash[:12]}...")

## 2) ربط كل كتلة بهاش التي قبلها، و 3) دالة التحقق
- `add_reading()` تنشئ كتلة جديدة وتضع فيها هاش آخر كتلة، والجهات المعتمدة فقط تستطيع الكتابة.
- تصنيف الكثافة: آمن (أقل من 2 شخص/م²)، مزدحم (2 إلى 4)، حرج (4 فأكثر).
- `is_valid()` تعيد حساب هاش كل كتلة وتتأكد من سلامة الروابط، وتحدد رقم الكتلة التي حدث فيها الخلل.

In [3]:
AUTHORIZED_AGENCIES = {"الأمن", "الصحة", "التشغيل"}
SYSTEM = "النظام"


def density_status(density):
    """تصنيف بسيط لسلامة الحشود (شخص لكل متر مربع)."""
    if density < 2:
        return "آمن"
    if density < 4:
        return "مزدحم"
    return "حرج"


class CrowdChain:
    def __init__(self):
        self.blocks = [Block(0, SYSTEM, {"note": "كتلة البداية - تم إنشاء السلسلة"},
                             previous_hash="0" * 64,
                             timestamp="2026-06-01T00:00:00+00:00")]

    @property
    def last_block(self):
        return self.blocks[-1]

    def add_reading(self, agency, site, people_count, area_m2):
        if agency not in AUTHORIZED_AGENCIES:
            raise PermissionError(f"'{agency}' ليست جهة معتمدة للكتابة في السجل")
        density = round(people_count / area_m2, 2)
        reading = {
            "site": site,
            "people_count": people_count,
            "area_m2": area_m2,
            "density_per_m2": density,
            "status": density_status(density),
        }
        block = Block(len(self.blocks), agency, reading, self.last_block.hash)
        self.blocks.append(block)
        return block

    def is_valid(self):
        """تعيد (True, رسالة) إذا كانت السلسلة سليمة، وإلا (False, سبب الخلل)."""
        for i, block in enumerate(self.blocks):
            if block.hash != block.compute_hash():
                return False, f"الكتلة {i}: تم تعديل بياناتها (الهاش المخزّن لا يطابق محتواها)"
            if i == 0:
                continue
            if block.previous_hash != self.blocks[i - 1].hash:
                return False, f"الكتلة {i}: الرابط مكسور (هاش السابق لا يطابق الكتلة {i-1})"
        return True, f"السلسلة سليمة - تم التحقق من {len(self.blocks)} كتل"

    def show(self):
        for b in self.blocks:
            print(b)


def print_check(chain):
    ok, msg = chain.is_valid()
    print(("✅ " if ok else "❌ تلاعب مكتشف: ") + msg)


def describe(reading):
    return (f"العدد: {reading['people_count']} | الكثافة: {reading['density_per_m2']} شخص/م² "
            f"| الحالة: {reading['status']}")

### نسخ موزعة
كل جهة تحتفظ بنسخة من السلسلة. حتى لو أعادت جهة بناء نسختها كاملة، فمقارنة آخر هاش بين النسخ تكشف النسخة المختلفة عن الأغلبية.

In [4]:
def compare_copies(copies):
    tips = {name: chain.last_block.hash for name, chain in copies.items()}
    votes = {}
    for h in tips.values():
        votes[h] = votes.get(h, 0) + 1
    majority = max(votes, key=votes.get)
    suspicious = [name for name, h in tips.items() if h != majority]
    return majority, suspicious

## 4) جرّب التلاعب وشاهد اكتشافه

### الخطوة 1: الجهات تسجّل قراءاتها في السلسلة المشتركة

In [5]:
chain = CrowdChain()
chain.add_reading("التشغيل", "جسر الجمرات - الدور 2", 9000, 3000)
chain.add_reading("الأمن",   "منى - البوابة 4",       5200, 1000)
chain.add_reading("الصحة",   "عرفات - المنطقة ب",     1500, 1000)
chain.add_reading("التشغيل", "مزدلفة - القطاع 7",     7600, 2000)
chain.show()
print()
print_check(chain)

# كل جهة تحتفظ بنسخة مطابقة
copies = {name: copy.deepcopy(chain) for name in AUTHORIZED_AGENCIES}

الكتلة 0 | كتلة البداية - تم إنشاء السلسلة
      الهاش: 0a042974b7b8...   السابق: 000000000000...
الكتلة 1 | الجهة: التشغيل | الموقع: جسر الجمرات - الدور 2 | الكثافة: 3.0 شخص/م² | الحالة: مزدحم
      الهاش: 0e024aa4e3dc...   السابق: 0a042974b7b8...
الكتلة 2 | الجهة: الأمن | الموقع: منى - البوابة 4 | الكثافة: 5.2 شخص/م² | الحالة: حرج
      الهاش: dccabc7a44a0...   السابق: 0e024aa4e3dc...
الكتلة 3 | الجهة: الصحة | الموقع: عرفات - المنطقة ب | الكثافة: 1.5 شخص/م² | الحالة: آمن
      الهاش: b93079c105df...   السابق: dccabc7a44a0...
الكتلة 4 | الجهة: التشغيل | الموقع: مزدلفة - القطاع 7 | الكثافة: 3.8 شخص/م² | الحالة: مزدحم
      الهاش: ec7366425cb2...   السابق: b93079c105df...

✅ السلسلة سليمة - تم التحقق من 5 كتل


### الخطوة 2: تلاعب بسيط
شخص يخفض قراءة "منى - البوابة 4" من حرج إلى آمن لإخفاء الازدحام.

In [6]:
target = chain.blocks[2]
print("قبل التعديل:", describe(target.reading))
target.reading["people_count"] = 1500
target.reading["density_per_m2"] = 1.5
target.reading["status"] = "آمن"
print("بعد التعديل:", describe(target.reading))
print()
print_check(chain)

قبل التعديل: العدد: 5200 | الكثافة: 5.2 شخص/م² | الحالة: حرج
بعد التعديل: العدد: 1500 | الكثافة: 1.5 شخص/م² | الحالة: آمن

❌ تلاعب مكتشف: الكتلة 2: تم تعديل بياناتها (الهاش المخزّن لا يطابق محتواها)


### الخطوة 3: مهاجم أذكى يعيد حساب هاش الكتلة المعدّلة
الكتلة نفسها تبدو سليمة الآن، لكن الكتلة التالية ما زالت تشير للهاش القديم.

In [7]:
target.hash = target.compute_hash()
print_check(chain)

❌ تلاعب مكتشف: الكتلة 3: الرابط مكسور (هاش السابق لا يطابق الكتلة 2)


### الخطوة 4: أقوى مهاجم يعيد بناء السلسلة كاملة في نسخته
نسخته وحدها تجتاز التحقق، لكن مقارنتها بنسخ الجهات الأخرى تكشفها: لا تستطيع جهة واحدة إعادة كتابة التاريخ وحدها.

In [8]:
forged = copies["الأمن"]
forged.blocks[2].reading.update({"people_count": 1500, "density_per_m2": 1.5, "status": "آمن"})
for i, b in enumerate(forged.blocks):
    if i > 0:
        b.previous_hash = forged.blocks[i - 1].hash
    b.hash = b.compute_hash()

print("نسخة جهة الأمن وحدها:")
print_check(forged)
majority, suspicious = compare_copies(copies)
print(f"\nآخر هاش لدى الأغلبية: {majority[:12]}...")
print("❌ النسخ المخالفة للأغلبية:", "، ".join(suspicious))

نسخة جهة الأمن وحدها:
✅ السلسلة سليمة - تم التحقق من 5 كتل

آخر هاش لدى الأغلبية: ec7366425cb2...
❌ النسخ المخالفة للأغلبية: الأمن


### الخطوة 5: جهة غير معتمدة تحاول الكتابة

In [9]:
try:
    CrowdChain().add_reading("مستخدم مجهول", "منى - البوابة 4", 10, 1000)
except PermissionError as e:
    print("🚫 مرفوض:", e)

🚫 مرفوض: 'مستخدم مجهول' ليست جهة معتمدة للكتابة في السجل


## حدود النموذج والتطوير المستقبلي
- النموذج يحاكي الشبكة داخل برنامج واحد. النظام الحقيقي يحتاج عُقدًا منفصلة لكل جهة وآلية إجماع (مثل PBFT المناسبة للشبكات المصرّح بها).
- إضافة توقيع رقمي لكل جهة، ليثبت هوية من سجّل القراءة وليس فقط أنها لم تتغير.
- ربط القراءات تلقائيًا بنموذج رؤية حاسوبية يعدّ الحشود من الكاميرات، بدل إدخالها يدويًا.